# Gemini ve LangChain ile LLM API'larını Çağırma Giriş 🦜🔗

Bu notebook'ta LangChain aracılığıyla LLM API'larını nasıl kullanacağınızı öğreneceksiniz. Örnek olarak Google'ın Gemini API'sını kullanacağız. Bu notebook'un sonunda, LangChain kullanarak API çağrıları yapmayı ve bunu neden yaptığımızı bileceksiniz.

## ⚙️ Kurulum

👉 Kurulum aşamasında oluşturduğumuz `.env` dosyasındaki ortam değişkenlerini yüklemek için aşağıdaki hücreyi çalıştırın:

In [1]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

True

👉 Hücrenin çıktısı "`True`" mu? Harika! Artık Gemini API ile kimlik doğrulaması yapmak için kullanılacak bir `GOOGLE_API_KEY` ortam değişkeni kurmuş olduk.

Eğer değilse, yardım isteyin.

## Basit Bir API Çağrısı Yapma

Bu notebook'ta şunların nasıl yapılacağını göstereceğiz:
1. Google'ın kendi kütüphanesini kullanarak API çağrısı yapma.
2. Aynı işlemi LangChain kullanarak yapma.

## Google Generative AI Kütüphanesini Kullanma

In [2]:
from google import genai

In [3]:
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the capital of France?",
)

`response` nesnesine bir göz atalım.

In [4]:
response.candidates[0].content.parts[0].text

'The capital of France is **Paris**.'

Gerçek cevabı nasıl alabileceğinizi görüyor musunuz?

Neyse ki, cevabı hemen almak için sadece `.text` özelliğini kullanabiliriz. Deneyin.

In [5]:
response.text

'The capital of France is **Paris**.'

Gemini cevaplarını Markdown formatında döndürür. Bunu kullanalım!

In [6]:
from IPython.display import Markdown
Markdown(response.text)

The capital of France is **Paris**.

Oluşturma parametrelerini de değiştirebilirsiniz. `google.genai` kullanarak bunu şu şekilde yaparsınız:

In [7]:
from google import genai
from google.genai import types # We need to import types for the config

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a social media post about how much you're learning about transformers.",
    config=types.GenerateContentConfig(
        max_output_tokens=200,
        temperature=1.0
    )
)

In [8]:
Markdown(response.text)

Here are a few options for a social media post about learning about transformers, playing with different tones and focuses. Choose the one that best fits your style!

---

**Option 1: Enthusiastic & General**

🤯 My brain is officially buzzing with all things #Transformers! 🤖 Seriously, diving deep into how these models work is mind-blowing. The attention mechanism alone is a game-changer. Feeling so much smarter with every paper I read and tutorial I watch. The future of AI feels more real than ever! #MachineLearning #DeepLearning #AI #NLP #LearningJourney

---

**Option 2: Slightly More Technical & Curious**

Currently on a deep dive into the world of #Transformers! 🧠 From self-attention to positional encoding, the elegance of these architectures is truly remarkable. It's fascinating to see how they've revolutionized fields like #NLP. Excited to keep unraveling their complexities and potential applications! #ArtificialIntelligence #DeepLearningModels

Harika. Ancak başka bir API denemek istediğinizi düşünün, örneğin OpenAI'nin veya Anthropic'in?

Onların dokümantasyonlarını incelemek ve tüm kodunuzu onların API'sini kullanacak şekilde yeniden yazmak zorunda kalırsınız. Tabii ki benzer olacaktır, ancak aynı olmayacaktır.

Neyse ki LangChain var!

## LangChain Kullanma 🦜🔗

Neden LangChain kullanırsınız?

1. **Model-Bağımsız Kod**

   LangChain, farklı LLM sağlayıcıları (Google, OpenAI, Anthropic, vb.) arasında minimal kod değişikliği ile geçiş yapmanızı sağlayan soyutlamalar sunar. Google API'sine doğrudan kod yazarsanız, sağlayıcı değiştirmek önemli ölçüde yeniden düzenleme gerektirir.

2. **Birleşik Arayüz**

   LangChain, altta yatan API'den bağımsız olarak farklı LLM sağlayıcıları arasında etkileşimleri standartlaştırır ve tutarlı yöntemler ile yanıt formatları sunar.

3. **Bileşenlerle Çalışabilirlik**

   LangChain'in zincir ve pipeline mimarisi, tüm alt yapıyı kendiniz halletmeden prompt, bellek ve erişim sistemlerini birleştiren karmaşık iş akışları oluşturmayı kolaylaştırır.

4. **Yerleşik Araçlar**

   LangChain, çıktı ayrıştırma, prompt şablonları ve kendiniz uygulamanız gereken diğer yardımcı araçları içerir.

[LangChain'in chat entegrasyonları listesi](https://docs.langchain.com/oss/python/integrations/chat)'ne gidin ve entegrasyon listesine bakın. Favori LLM sağlayıcınızı bulabiliyor musunuz?

Kodumuzda `chat_models.ChatGoogleGenerativeAI` kullanmak istemiyoruz çünkü bu özellikle Gemini için yapılmış. LLM'yi değiştirmek istersek, modeli başlatma şeklimizi değiştirmek zorunda kalırız. Neyse ki LangChain bir modeli başlatmak için daha genel bir yol sunar.

Gemini'yi tekrar kullanalım, ancak şimdi LangChain'in genel Chat Models'ini kullanarak.

👉 [LangChain'in "Models" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/models) sayfasına gidin ve Gemini kullanarak bir chat modelinin nasıl başlatılacağını bulun.

İpuçları:
1. Hemen "Basic Usage" bölümüne gidin.
2. Kullanmak istediğiniz modeli seçerek doğru dokümantasyonu hemen görebilirsiniz.

In [9]:
import os

from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = api_key



model = init_chat_model("google_genai:gemini-2.5-flash-lite")

Modelin en temel kullanımı sadece `.invoke()` metodunu kullanmaktır:

In [10]:
response = model.invoke("Why do parrots talk?")

Yanıta bir göz atalım. Nesnenin tüm öznitelik ve metodlarını içeren `__dict__`'ini güzel şekilde yazdırmak için `pprint()` kullanıyoruz.

In [11]:
from pprint import pprint
pprint(response.__dict__)

{'additional_kwargs': {},
 'content': 'Parrots talk primarily due to their remarkable **vocal learning '
            'abilities**, combined with strong **social drives** and a '
            "**cognitive capacity** for understanding and using language. It's "
            "not just about mimicry; it's a complex interplay of factors.\n"
            '\n'
            "Here's a breakdown of the main reasons:\n"
            '\n'
            '**1. Vocal Learning:**\n'
            '\n'
            '*   **Exceptional Mimicry:** This is the most obvious reason. '
            'Parrots have highly developed vocal tracts and a brain structure '
            'that allows them to learn and reproduce a wide range of sounds, '
            'including human speech. They can imitate not just words but also '
            'intonation, pitch, and even accents.\n'
            '*   **Neural Basis:** Studies have shown that parrots have a '
            'unique "song system" in their brains, similar to what is fou

Cevabı çıkarın ve görüntüleyin. Markdown formatında olduğunu unutmayın, bu yüzden güzel görünmesini sağlayabilirsiniz.

In [12]:
response.text

'Parrots talk primarily due to their remarkable **vocal learning abilities**, combined with strong **social drives** and a **cognitive capacity** for understanding and using language. It\'s not just about mimicry; it\'s a complex interplay of factors.\n\nHere\'s a breakdown of the main reasons:\n\n**1. Vocal Learning:**\n\n*   **Exceptional Mimicry:** This is the most obvious reason. Parrots have highly developed vocal tracts and a brain structure that allows them to learn and reproduce a wide range of sounds, including human speech. They can imitate not just words but also intonation, pitch, and even accents.\n*   **Neural Basis:** Studies have shown that parrots have a unique "song system" in their brains, similar to what is found in songbirds, which is crucial for vocal learning. They have specialized brain regions that are highly plastic, meaning they can change and adapt to new sounds.\n\n**2. Social Communication and Bonding:**\n\n*   **Flock Behavior:** In the wild, parrots are 

In [13]:
from IPython.display import Markdown, display
display(Markdown(response.content))

Parrots talk primarily due to their remarkable **vocal learning abilities**, combined with strong **social drives** and a **cognitive capacity** for understanding and using language. It's not just about mimicry; it's a complex interplay of factors.

Here's a breakdown of the main reasons:

**1. Vocal Learning:**

*   **Exceptional Mimicry:** This is the most obvious reason. Parrots have highly developed vocal tracts and a brain structure that allows them to learn and reproduce a wide range of sounds, including human speech. They can imitate not just words but also intonation, pitch, and even accents.
*   **Neural Basis:** Studies have shown that parrots have a unique "song system" in their brains, similar to what is found in songbirds, which is crucial for vocal learning. They have specialized brain regions that are highly plastic, meaning they can change and adapt to new sounds.

**2. Social Communication and Bonding:**

*   **Flock Behavior:** In the wild, parrots are highly social creatures that live in large flocks. They use a variety of vocalizations to communicate within the flock, including contact calls to maintain cohesion, warnings of danger, and greetings.
*   **Social Bonds:** Talking can be a way for parrots to strengthen their bonds with their flock members, including their human "flock" (owners). By mimicking sounds they hear from their social group, they are essentially participating in the group's communication.
*   **Attention Seeking:** Parrots are intelligent and often crave attention. Mimicking human speech is a highly effective way to get their owners' attention and interact with them.

**3. Cognitive Abilities:**

*   **Intelligence:** Parrots are among the most intelligent bird species. They possess problem-solving skills, can understand cause and effect, and have good memories.
*   **Association and Context:** While they might not understand the literal meaning of every word like humans do, many parrots can learn to associate words and phrases with specific objects, actions, or contexts. For example, a parrot might learn to say "hello" when someone enters the room or "cracker" when it wants a treat.
*   **Intentional Communication:** Some studies suggest that certain parrots can use learned vocalizations with a degree of intentionality, even if it's not full human-level comprehension. They might be trying to achieve a specific outcome, like getting food or attention.

**4. Environmental Factors:**

*   **Stimulation:** Parrots kept in environments with a lot of human interaction and speech are more likely to be exposed to and learn vocalizations. A stimulating environment encourages them to vocalize and interact.
*   **Boredom and Frustration:** Sometimes, when parrots are bored or frustrated, they may vocalize excessively, and mimicking sounds can be a way for them to express themselves.

**In summary, parrots talk because they are:**

*   **Physically capable** of producing complex sounds.
*   **Mentally wired** for vocal learning.
*   **Socially motivated** to communicate and bond.
*   **Intelligent enough** to make associations and potentially use vocalizations with some degree of intention.

It's a fascinating testament to their evolutionary adaptations and cognitive prowess that these beautiful birds can share our vocalizations and bring so much joy and interaction into our lives.

Modelin temperature değerini `.temperature` özniteliğine erişerek kontrol edebilirsiniz. Deneyin:

In [14]:
model.temperature

0.7

Modeli kullanmadan önce, özniteliklere yeni değerler atayarak oluşturma parametrelerini de ayarlayabiliriz.

Daha önce Google'ın kütüphanesini kullanarak sosyal medya gönderisi yazmak için yaptığımızın eşdeğerini kodlamaya çalışın.

> _Not_: Normal olarak modelin `max_output_tokens` değerini ayarlayabilmemiz gerekir (modeli başlatırken veya daha sonra özniteliği değiştirerek). _langchain_google_genai_'nin mevcut sürümü (4.1.1) bir [hataya](https://github.com/langchain-ai/langchain-google/issues/1454) sahip ve bu çalışmıyor. Geçici çözüm? `max_output_tokens`'ı `.invoke()` metodunun bir parametresi olarak ayarlayın.

In [16]:
# Set the maximum number of output tokens to 200
model.max_output_tokens = 200

# Set the temperature to 1.0

model.temperature = 1.0
# Generate a response with the new settings

response = model.invoke("Veri analitiği ve yapay zeka alanında kariyer yapmak isteyenler için heyecan verici, emojilerle süslü bir LinkedIn gönderisi hazırla.")
# Display the response

Markdown(response.text)


Harika bir LinkedIn gönderisi hazırlayalım! İşte veri analitiği ve yapay zeka alanında kariyer yapmak isteyenler için heyecan verici, emojilerle süslü bir seçenek:

---

🚀 Veri Deryasında Bir Yolculuğa Hazır mısınız? Yapay Zeka Dünyası Sizi Çağırıyor! 🤖✨

Herkese merhaba! 👋 Eğer siz de makinelerin dilini anlamak, verilerin gizemini çözmek ve geleceği şekillendiren teknolojilerin bir parçası olmak istiyorsanız, doğru yerdesiniz! 🎯

Veri analitiği 📊 ve yapay zeka 🧠, günümüzün en hızlı gelişen ve en heyecan verici alanlarından biri. Bu dinamik dünyada kariyer yapmak isteyenler için inanılmaz fırsatlar var! 💪

**Neden bu alanlar bu kadar büyüleyici? 🤔**

*   **Sorunları Çözme Gücü:** Verilerle olayları daha iyi anlarız, öngörülerde bulunuruz ve stratejiler geliştiririz. 💡
*   **Yenilikçilik Rüzgarı:** Yapay zeka, her sektörde devrim yaratıyor. Otomotivden sağlığa, finanstan sanata kadar her yerde iz bırakıyor! 🚗🏥💰🎨
*   **Sürekli Öğrenme:** Bu alanlar o kadar hızlı gelişiyor ki, her gün yeni bir şey öğrenmek kaçınılmaz ve eğlenceli! 📚🤓
*   **Geleceğe Yatırım:** Yapay zeka ve veri analistleri, geleceğin en çok aranan profesyonelleri arasında yer alıyor. 🌟

**Peki, bu heyecan verici yolculuğa nasıl başlayabilirsiniz? 🗺️**

1.  **Temelleri Sağlam Atın:** Matematik, istatistik ve programlama dilleri (Python 🐍, R 📈) gibi temel becerileri edinin.
2.  **Bol Bol Pratik Yapın:** Kaggle gibi platformlarda veri setleriyle oynayın, projeler geliştirin. 👩‍💻👨‍💻
3.  **Kurslara ve Eğitimlere Katılın:** Online platformlar (Coursera, Udemy, edX) ve üniversiteler harika kaynaklar sunuyor. 🎓
4.  **Topluluklara Dahil Olun:** Konferanslara katılın, webinarları izleyin, LinkedIn'deki ilgili gruplarda yer alın. 🤝
5.  **Merakınızı Kaybetmeyin:** En önemlisi, öğrenme isteğiniz ve problem çözme tutkunuz olsun! 🔥

Bu alanda kariyer yapmayı düşünen herkesi tebrik ediyorum! 🎉 Birlikte daha akıllı bir gelecek inşa edebiliriz. 🌐

Bu yolculukta sizin de hikayelerinizi, ipuçlarınızı veya sorularınızı duymak isterim! 👇 Yorumlarda buluşalım!

#VeriAnalizi #YapayZeka #Kariyer #Gelecek #Teknoloji #MachineLearning #DeepLearning #DataScience #Python #R #Analitik #KariyerFırsatları #Innovasyon #Öğrenme

---

**Ek İpuçları:**

*   Gönderiyi yayınlamadan önce görsel olarak ilgi çekici bir resim veya kısa bir video ekleyebilirsiniz. Veri görselleştirmeleri, yapay zeka temalı grafikler veya kod ekran görüntüsü gibi şeyler harika olur.
*   Kendi deneyimlerinizden kısa bir anekdot eklemek, gönderiyi daha kişisel ve samimi hale getirebilir.
*   Gönderiyi yayınladıktan sonra gelen yorumlara mutlaka yanıt verin ve etkileşim kurun.

Umarım beğenirsiniz! Başarılar dilerim! ✨

Bunun avantajı? Bu LangChain Chat Model birçok başka API'yi destekleyebilir.

Başka bir modele geçmek için değiştirmeniz gereken tek şeyler:
1. Diğer model için bir API anahtarı alın ve kodunuzda tanımlayın.
2. Modeli başlatırken model ve sağlayıcıyı değiştirin.

### Çoklu Mesajlar

`.invoke()` fonksiyonunu sadece tek bir mesajla kullanmak biraz kısıtlayıcı.

Şu gibi birden fazla mesaj sağlayabilirsiniz:
- `SystemMessage` veya sistem mesajları: modelin nasıl davranacağını söylemek için
- `HumanMessage` veya Kullanıcı mesajları: kullanıcıdan gelen girdi
- `AIMessage` veya Asistan mesajları: modelden gelen yanıt

Bir sosyal medya yazarı yapalım.

Modele nasıl davranacağını açıklayan bir sistem mesajı göndereceğiz. Sonra kullanıcı mesajında, kendimizi sadece yazacağı konuyu vermekle sınırlayabiliriz.

Bunu nasıl yapacağınızı öğrenmek için [LangChain'in "Messages" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/messages)'na bakın.

Sistem mesajı için ilhama mı ihtiyacınız var? İşte başlamanız için temel bir talimat:

```python
"""Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
Gönderilerin maksimum 200 karakter uzunluğundadır.
Her zaman emoji kullanırsın.
"""
```

In [17]:
# Import the necessary classes
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Create a list of messages

messages = [
    SystemMessage(content="""
        Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
        Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
        Gönderilerin maksimum 200 karakter uzunluğundadır.
        Her zaman emoji kullanırsın.
    """),
    HumanMessage(content="Konu: Büyük Dil Modelleri (LLM) nasıl çalışır?")
]

# Generate a response using the list of messages

response = model.invoke(messages)

# Display the response

print(f"AI Yazarı: {response.content}")


AI Yazarı: Büyük Dil Modelleri (LLM) nasıl mı çalışır? 🤯 Verileriyle "büyük" lafı yeni çıkmış! 📚 Onlarca terabayt veri üzerinde eğitilen bu modeller, metin oluşturabilir, çeviri yapabilir ve daha fazlasını! 🚀

**Hemen öğren, LLM'lerin sırrını çöz!** 👇


🏁 Tebrikler! Artık LangChain kullanarak çoklu mesajlarla temel prompt yazma konusunda uzmanlaştınız.